# Evaluation with RAGAS and Advanced Retrieval Methods Using LangChain

We're going to be leveraging the [RAGAS](https://docs.ragas.io/en/stable/) framework for our evaluations.

We're also going to explore a few more powerful Retrieval Systems that can potentially improve the quality of our generations!

## Imports

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
from langchain.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_docling import DoclingLoader
from langchain.prompts import ChatPromptTemplate
from operator import itemgetter
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnableLambda, RunnablePassthrough
from langchain.output_parsers import ResponseSchema
from langchain.output_parsers import StructuredOutputParser

## Data Collection

We will be using Arxiv [A Survey on LLM-as-a-Judge](https://arxiv.org/abs/2411.15594) paper as our context.

In [9]:
source = "../data/llm-as-judge.pdf"  # file path or URL
loader = DoclingLoader(file_path=source)
base_docs = loader.load()

2025-09-18 14:47:01,694 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-09-18 14:47:01,709 - INFO - Going to convert document batch...
2025-09-18 14:47:01,710 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e647edf348883bed75367b22fbe60347
2025-09-18 14:47:01,710 - INFO - Accelerator device: 'mps'
2025-09-18 14:47:04,015 - INFO - Accelerator device: 'mps'
2025-09-18 14:47:04,923 - INFO - Accelerator device: 'mps'
2025-09-18 14:47:05,507 - INFO - Processing document llm-as-judge.pdf
2025-09-18 14:47:38,342 - INFO - Finished converting document llm-as-judge.pdf in 36.65 sec.
Token indices sequence length is longer than the specified maximum sequence length for this model (1419 > 512). Running this sequence through the model will result in indexing errors


In [10]:
len(base_docs)

294

## Creating an Index

Let's use a naive index creation strategy of just using `RecursiveCharacterTextSplitter` on our documents and embedding each into our `VectorStore` using `BAAI/bge-small-en`.

In [11]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512)
docs = text_splitter.split_documents(base_docs)

In [12]:
len(docs)

823

In [13]:
model_name = "BAAI/bge-small-en"
model_kwargs = {"device": "mps"}
encode_kwargs = {"normalize_embeddings": True}
embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs
)

vectorstore = FAISS.from_documents(docs, embeddings)

/var/folders/78/86ckn1md5y5drxprldlf315c0000gn/T/ipykernel_26736/1088778382.py:4: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceBgeEmbeddings(
2025-09-18 14:47:42,730 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en
2025-09-18 14:47:49,353 - INFO - Loading faiss.
2025-09-18 14:47:49,436 - INFO - Successfully loaded faiss.


In [14]:
print(max([len(chunk.page_content) for chunk in docs]))

511


In [15]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k" : 5})

In [16]:
base_retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceBgeEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x474d745f0>, search_kwargs={'k': 5})

Now to give it a test...

In [17]:
question = "What is LLM-as-a-Judge?"

relevant_docs = base_retriever.invoke(question)

In [18]:
len(relevant_docs)

5

In [19]:
relevant_docs

[Document(id='853bff69-8898-4df6-bb8f-03a21be8a8c1', metadata={'source': '../data/llm-as-judge.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/786', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 37, 'bbox': {'l': 45.46, 't': 323.418, 'r': 441.702, 'b': 171.288, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 1210]}]}], 'headings': ['9.2 LLM-as-a-Judge for Data Annotation'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 4349892626816481402, 'filename': 'llm-as-judge.pdf'}}}, page_content='In contrast, LLM-as-a-judge is a general technique where you use LLM to approximate human labeling. When you ask an LLM to assess qualities like "faithfulness to source," "correctness," or "helpfulness," you define what these terms mean in the evaluation prompt and rely on the semantic relationships the LLM learned from training data. Despite its w

## Creating a Retrieval Augmented Generation Prompt

Now we can set up a prompt template that will be used to provide the LLM with the necessary contexts, user query, and instructions!

In [20]:
template = """Answer the question based only on the following context. If you cannot answer the question with the context, please respond with 'I don't know':

### CONTEXT
{context}

### QUESTION
Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [21]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Answer the question based only on the following context. If you cannot answer the question with the context, please respond with 'I don't know':\n\n### CONTEXT\n{context}\n\n### QUESTION\nQuestion: {question}\n"), additional_kwargs={})])

## Setup LLMs

For best results, it is recommended to use state-of-the-art models like GPT-4 as `answer_generation_llm` to generate high-quality ground truth (reference answers), which can then be used to assess the accuracy and relevance of responses produced by the question-answering model i.e. `question_generation_llm`.

I will use Ollama models to get things done, without spending any token cost ;)

In [22]:
question_generation_llm = init_chat_model(model="gemma3:1b", model_provider="ollama", temperature=0) # your Q&A model
answer_generation_llm = init_chat_model(model="gemma3:4b", model_provider="ollama", temperature=0) # your SOTA model like GPT-4 latest version or Deepseek

## Setting Up our Basic QA Chain

Now we can instantiate our basic RAG chain!

We'll follow *exactly* the chain we made on Tuesday to keep things simple for now - if you need a refresher on what it looked like - check out last week's notebook!

In [23]:
retrieval_augmented_qa_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | base_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": prompt | question_generation_llm, "context": itemgetter("context")}
)

Let's test it out!

In [24]:
result = retrieval_augmented_qa_chain.invoke({"question" : question})

print(result)

2025-09-18 14:47:52,897 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


{'response': AIMessage(content='LLM-as-a-judge is a general technique where you use LLM to approximate human labeling.', additional_kwargs={}, response_metadata={'model': 'gemma3:1b', 'created_at': '2025-09-18T12:47:53.098516Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2318090458, 'load_duration': 1278435708, 'prompt_eval_count': 2120, 'prompt_eval_duration': 833818875, 'eval_count': 23, 'eval_duration': 204922708, 'model_name': 'gemma3:1b'}, id='run--c5872e48-bb2e-4b86-b563-b8ac2784839d-0', usage_metadata={'input_tokens': 2120, 'output_tokens': 23, 'total_tokens': 2143}), 'context': [Document(id='853bff69-8898-4df6-bb8f-03a21be8a8c1', metadata={'source': '../data/llm-as-judge.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/786', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 37, 'bbox': {'l': 45.46, 't': 323.418, 'r': 441.702, 

## Ground Truth Dataset Creation Using SOTA models like GPT-4, but we will use `"gemma3:4b"` ;)

This process might take long time to run, depending on the number of questions you want to generate across chunks.

The basic idea is that we can use LangChain to create questions based on our contexts, and then answer those questions.

In [25]:
question_schema = ResponseSchema(
    name="question",
    description="a question about the context."
)

question_response_schemas = [
    question_schema,
]

In [26]:
question_output_parser = StructuredOutputParser.from_response_schemas(question_response_schemas)
format_instructions = question_output_parser.get_format_instructions()

In [27]:
bare_prompt_template = "{content}"
bare_template = ChatPromptTemplate.from_template(template=bare_prompt_template)

In [28]:
qa_template = """\
You are an AI Expert. For each context, create a question that is specific to the context. Avoid creating generic or general questions.

question: a question about the context.

Format the output as JSON with the following keys:
question

context: {context}
"""

prompt_template = ChatPromptTemplate.from_template(template=qa_template)

messages = prompt_template.format_messages(
    context=docs[5],
    format_instructions=format_instructions
)

question_generation_chain = bare_template | question_generation_llm

response = question_generation_chain.invoke({"content" : messages})
output_dict = question_output_parser.parse(response.content)

2025-09-18 14:47:53,348 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [29]:
for k, v in output_dict.items():
  print(k)
  print(v)

question
Considering the specific data format and structure presented in the page content, what are the primary challenges in ensuring consistent metadata extraction and interpretation across different versions of the LLM-as-judge dataset?


In [30]:
from tqdm import tqdm

qac_triples = []

for text in tqdm(docs[3:7]):
  messages = prompt_template.format_messages(
      context=text,
      format_instructions=format_instructions
  )
  response = question_generation_chain.invoke({"content" : messages})
  try:
    output_dict = question_output_parser.parse(response.content)
  except Exception as e:
    continue
  output_dict["context"] = text
  qac_triples.append(output_dict)

100%|██████████| 4/4 [00:02<00:00,  1.45it/s]


In [31]:
qac_triples

[{'question': "Considering the specific data structure within the PDF document, what types of textual analysis are most likely to be performed by an LLM-as-a-Judge, and how might the LLM's assessment process differ from traditional human evaluation methods based on the provided document's content?",
  'context': Document(metadata={'source': '../data/llm-as-judge.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/11', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 45.496, 't': 436.504, 'r': 441.691, 'b': 275.279, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 1556]}]}], 'headings': ['ABSTRACT'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 4349892626816481402, 'filename': 'llm-as-judge.pdf'}}}, page_content='Accurate and consistent evaluation is crucial for decision-making across numerous fields, yet it remains a chal

In [32]:
answer_schema = ResponseSchema(
    name="answer",
    description="an answer to the question"
)

answer_response_schemas = [
    answer_schema,
]

answer_output_parser = StructuredOutputParser.from_response_schemas(answer_response_schemas)
format_instructions = answer_output_parser.get_format_instructions()

qa_template = """\
You are an AI Expert. For each question and context, create an answer.

answer: a answer about the context.

Format the output as JSON with the following keys:
answer

question: {question}
context: {context}
"""

prompt_template = ChatPromptTemplate.from_template(template=qa_template)

messages = prompt_template.format_messages(
    context=qac_triples[0]["context"],
    question=qac_triples[0]["question"],
    format_instructions=format_instructions
)

answer_generation_chain = bare_template | answer_generation_llm

response = answer_generation_chain.invoke({"content" : messages})
output_dict = answer_output_parser.parse(response.content)

2025-09-18 14:48:18,768 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [33]:
for k, v in output_dict.items():
  print(k)
  print(v)

answer
Given the context, an LLM-as-a-Judge would likely perform several types of textual analysis. Primarily, it would focus on identifying key claims, arguments, and evidence presented within the PDF document. Due to the nature of LLMs, it could also assess the consistency and coherence of these elements, flagging contradictions or ambiguities. Furthermore, the LLM could evaluate the document's overall structure and logical flow. 

Differing from traditional human evaluation, the LLM's assessment would be far more scalable and objective. Humans are susceptible to biases and subjective interpretations. The LLM, however, would apply consistent criteria based on its training data, providing a standardized evaluation. This removes the variability inherent in human judgment, allowing for rapid and large-scale assessments. The LLM wouldn't rely on nuanced understanding or contextual awareness in the same way a human would, focusing instead on quantifiable metrics related to the presence an

In [34]:
for triple in tqdm(qac_triples):
  messages = prompt_template.format_messages(
      context=triple["context"],
      question=triple["question"],
      format_instructions=format_instructions
  )
  response = answer_generation_chain.invoke({"content" : messages})
  try:
    output_dict = answer_output_parser.parse(response.content)
  except Exception as e:
    continue
  triple["answer"] = output_dict["answer"]

100%|██████████| 4/4 [00:10<00:00,  2.55s/it]


In [35]:
qac_triples

[{'question': "Considering the specific data structure within the PDF document, what types of textual analysis are most likely to be performed by an LLM-as-a-Judge, and how might the LLM's assessment process differ from traditional human evaluation methods based on the provided document's content?",
  'context': Document(metadata={'source': '../data/llm-as-judge.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/11', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 45.496, 't': 436.504, 'r': 441.691, 'b': 275.279, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 1556]}]}], 'headings': ['ABSTRACT'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 4349892626816481402, 'filename': 'llm-as-judge.pdf'}}}, page_content='Accurate and consistent evaluation is crucial for decision-making across numerous fields, yet it remains a chal

In [36]:
import pandas as pd
from datasets import Dataset

ground_truth_qac_set = pd.DataFrame(qac_triples)
ground_truth_qac_set["context"] = ground_truth_qac_set["context"].map(lambda x: str(x.page_content))
ground_truth_qac_set = ground_truth_qac_set.rename(columns={"answer" : "ground_truth"})

eval_dataset = Dataset.from_pandas(ground_truth_qac_set)

In [37]:
ground_truth_qac_set

,question,context,ground_truth
0,Considering the specific data structure within...,Accurate and consistent evaluation is crucial ...,"Given the context, an LLM-as-a-Judge would lik..."
1,What specific types of complex tasks are curre...,where LLMs are employed as evaluators for comp...,The context indicates that LLMs are currently ...
2,Considering the specific data format and struc...,challenge that requires careful design and sta...,The primary challenge in ensuring consistent m...
3,Considering the specific challenges of maintai...,"reliability, including improving consistency, ...",The context highlights the prioritization of r...


In [38]:
eval_dataset

Dataset({
    features: ['question', 'context', 'ground_truth'],
    num_rows: 4
})

In [39]:
eval_dataset[0]

{'question': "Considering the specific data structure within the PDF document, what types of textual analysis are most likely to be performed by an LLM-as-a-Judge, and how might the LLM's assessment process differ from traditional human evaluation methods based on the provided document's content?",
 'context': 'Accurate and consistent evaluation is crucial for decision-making across numerous fields, yet it remains a challenging task due to inherent subjectivity, variability, and scale. Large Language Models (LLMs) have achieved remarkable success across diverse domains, leading to the emergence of "LLM-as-a-Judge," where LLMs are employed as evaluators for complex tasks. With their ability to process diverse data types and provide scalable and flexible assessments, LLMs present a compelling alternative to',
 'ground_truth': "Given the context, an LLM-as-a-Judge would likely perform several types of textual analysis. Primarily, it would focus on identifying key claims, arguments, and ev

In [40]:
eval_dataset.to_csv("../data/groundtruth_eval_dataset.csv")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 129.13ba/s]


5318

## Evaluation

### Evaluating RAG Pipelines

If you skipped ahead and need to load the `.csv` directly - uncomment the code below.

If you're using Colab to do this notebook - please ensure you add it to your session files.

In [41]:
# from datasets import Dataset
# eval_dataset = Dataset.from_csv("../data/groundtruth_eval_dataset.csv")

In [42]:
eval_dataset

Dataset({
    features: ['question', 'context', 'ground_truth'],
    num_rows: 4
})

### Evaluation Using RAGAS

Now we can evaluate using RAGAS!

The set-up is - we simply need to create a dataset with our generated answers and our contexts, and then evaluate using the framework.

In [43]:
from ragas.metrics import (
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision,
    answer_correctness,
    answer_similarity
)

from ragas import evaluate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [44]:
eval_dataset[0]

{'question': "Considering the specific data structure within the PDF document, what types of textual analysis are most likely to be performed by an LLM-as-a-Judge, and how might the LLM's assessment process differ from traditional human evaluation methods based on the provided document's content?",
 'context': 'Accurate and consistent evaluation is crucial for decision-making across numerous fields, yet it remains a challenging task due to inherent subjectivity, variability, and scale. Large Language Models (LLMs) have achieved remarkable success across diverse domains, leading to the emergence of "LLM-as-a-Judge," where LLMs are employed as evaluators for complex tasks. With their ability to process diverse data types and provide scalable and flexible assessments, LLMs present a compelling alternative to',
 'ground_truth': "Given the context, an LLM-as-a-Judge would likely perform several types of textual analysis. Primarily, it would focus on identifying key claims, arguments, and ev

In [45]:
for row in tqdm(eval_dataset):
    print(row)

100%|██████████| 4/4 [00:00<00:00, 4650.00it/s]

{'question': "Considering the specific data structure within the PDF document, what types of textual analysis are most likely to be performed by an LLM-as-a-Judge, and how might the LLM's assessment process differ from traditional human evaluation methods based on the provided document's content?", 'context': 'Accurate and consistent evaluation is crucial for decision-making across numerous fields, yet it remains a challenging task due to inherent subjectivity, variability, and scale. Large Language Models (LLMs) have achieved remarkable success across diverse domains, leading to the emergence of "LLM-as-a-Judge," where LLMs are employed as evaluators for complex tasks. With their ability to process diverse data types and provide scalable and flexible assessments, LLMs present a compelling alternative to', 'ground_truth': "Given the context, an LLM-as-a-Judge would likely perform several types of textual analysis. Primarily, it would focus on identifying key claims, arguments, and evid

In [46]:
eval_dataset[0]

{'question': "Considering the specific data structure within the PDF document, what types of textual analysis are most likely to be performed by an LLM-as-a-Judge, and how might the LLM's assessment process differ from traditional human evaluation methods based on the provided document's content?",
 'context': 'Accurate and consistent evaluation is crucial for decision-making across numerous fields, yet it remains a challenging task due to inherent subjectivity, variability, and scale. Large Language Models (LLMs) have achieved remarkable success across diverse domains, leading to the emergence of "LLM-as-a-Judge," where LLMs are employed as evaluators for complex tasks. With their ability to process diverse data types and provide scalable and flexible assessments, LLMs present a compelling alternative to',
 'ground_truth': "Given the context, an LLM-as-a-Judge would likely perform several types of textual analysis. Primarily, it would focus on identifying key claims, arguments, and ev

In [47]:
def create_ragas_dataset(rag_pipeline, eval_dataset):
  rag_dataset = []
  for row in tqdm(eval_dataset):
    answer = rag_pipeline.invoke({"question" : row["question"]})
    rag_dataset.append(
        {"question" : row["question"],
         "answer" : answer["response"].content,
         "contexts" : [context.page_content for context in answer["context"]],
         "reference" : row["ground_truth"]
         }
    )
  rag_df = pd.DataFrame(rag_dataset)
  rag_eval_dataset = Dataset.from_pandas(rag_df)
  return rag_eval_dataset

def evaluate_ragas_dataset(ragas_dataset):
  result = evaluate(
    dataset = ragas_dataset,
    metrics=[
        context_precision,
        faithfulness,
        answer_relevancy,
        context_recall,
        answer_correctness,
        answer_similarity
    ],
    llm=answer_generation_llm,
    embeddings=embeddings
  )
  return result

Lets create our dataset first:

In [48]:
basic_qa_ragas_dataset = create_ragas_dataset(retrieval_augmented_qa_chain, eval_dataset)

100%|██████████| 4/4 [00:09<00:00,  2.43s/it]


In [49]:
basic_qa_ragas_dataset

Dataset({
    features: ['question', 'answer', 'contexts', 'reference'],
    num_rows: 4
})

In [50]:
basic_qa_ragas_dataset[0]

{'question': "Considering the specific data structure within the PDF document, what types of textual analysis are most likely to be performed by an LLM-as-a-Judge, and how might the LLM's assessment process differ from traditional human evaluation methods based on the provided document's content?",
 'answer': 'Based on the context, here’s a breakdown of the most likely textual analysis performed by an LLM-as-a-Judge, and how it might differ from traditional human evaluation:\n\n**Likely Textual Analysis:**\n\n*   **Quality Assessment:** The document highlights that LLM-as-a-judge is used to assess qualities like “faithfulness to source,” “correctness,” and “helpfulness.” This strongly suggests the LLM will be focused on evaluating the *quality* of the generated text.\n*   **Formula Extraction & Analysis:** The document explicitly mentions "formula" and "text" as key elements. This implies the LLM will likely perform analysis of the text\'s formula, potentially identifying patterns or r

Save it for later:

In [51]:
basic_qa_ragas_dataset.to_csv("../data/basic_qa_ragas_dataset.csv")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 507.36ba/s]


15644

In [52]:
basic_qa_ragas_dataset[0]

{'question': "Considering the specific data structure within the PDF document, what types of textual analysis are most likely to be performed by an LLM-as-a-Judge, and how might the LLM's assessment process differ from traditional human evaluation methods based on the provided document's content?",
 'answer': 'Based on the context, here’s a breakdown of the most likely textual analysis performed by an LLM-as-a-Judge, and how it might differ from traditional human evaluation:\n\n**Likely Textual Analysis:**\n\n*   **Quality Assessment:** The document highlights that LLM-as-a-judge is used to assess qualities like “faithfulness to source,” “correctness,” and “helpfulness.” This strongly suggests the LLM will be focused on evaluating the *quality* of the generated text.\n*   **Formula Extraction & Analysis:** The document explicitly mentions "formula" and "text" as key elements. This implies the LLM will likely perform analysis of the text\'s formula, potentially identifying patterns or r

And finally - evaluate how it did!

In [53]:
basic_qa_result = evaluate_ragas_dataset(basic_qa_ragas_dataset)

Evaluating:  12%|█▎        | 3/24 [00:01<00:08,  2.42it/s]2025-09-18 14:48:51,758 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:48:54,458 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:49:00,415 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
Evaluating:  21%|██        | 5/24 [00:14<01:07,  3.55s/it]2025-09-18 14:49:04,110 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:49:04,563 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:49:05,431 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:49:06,481 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:49:06,946 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
Evaluating:  25%|██▌       | 6/24 [00:19<01:09,  3.86s/it]2025-09-18

In [54]:
basic_qa_result

{'context_precision': nan, 'faithfulness': 0.3542, 'answer_relevancy': 0.2337, 'context_recall': 1.0000, 'answer_correctness': 0.5170, 'answer_similarity': 0.7954}

**Note:** While I was using Ollama model locally for evaluation, you can observe couple of `TimeoutError()`, maybe that's the reason we got inconsistent values across `context_precision` and `answer_relevancy`.

### Testing Other Retrievers

Now we can test our how changing our Retriever impacts our RAGAS evaluation!

We'll build this simple qa_chain factory to create standardized qa_chains where the only different component will be the retriever.

In [55]:
def create_qa_chain(retriever):
  created_qa_chain = (
    {"context": itemgetter("question") | retriever,
     "question": itemgetter("question")
    }
    | RunnablePassthrough.assign(
        context=itemgetter("context")
      )
    | {
         "response": prompt | question_generation_llm,
         "context": itemgetter("context"),
      }
  )

  return created_qa_chain

#### Parent Document Retriever

One of the easier ways we can imagine improving a retriever is to embed our documents into small chunks, and then retrieve a significant amount of additional context that "surrounds" the found context.

The basic outline of this retrieval method is as follows:

1. Obtain User Question
2. Retrieve child documents using Dense Vector Retrieval
3. Merge the child documents based on their parents. If they have the same parents - they become merged.
4. Replace the child documents with their respective parent documents from an in-memory-store.
5. Use the parent documents to augment generation.

In [56]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

base_docs_text = [x.page_content for x in base_docs]
vectorstore = FAISS.from_texts(base_docs_text, embeddings)
store = InMemoryStore()

In [57]:
parent_document_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [58]:
parent_document_retriever.add_documents(base_docs)

Let's create, test, and then evaluate our new chain!

In [59]:
parent_document_retriever_qa_chain = create_qa_chain(parent_document_retriever)

In [60]:
parent_document_retriever_qa_chain.invoke({"question" : question})["response"].content

2025-09-18 14:52:21,828 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


'LLM-as-a-Judge is a general technique where you use LLM to approximate human labeling.'

In [61]:
pdr_qa_ragas_dataset = create_ragas_dataset(parent_document_retriever_qa_chain, eval_dataset)

100%|██████████| 4/4 [00:04<00:00,  1.04s/it]


In [62]:
pdr_qa_ragas_dataset.to_csv("../data/pdr_qa_ragas_dataset.csv")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 490.91ba/s]


12664

In [63]:
pdr_qa_result = evaluate_ragas_dataset(pdr_qa_ragas_dataset)

Evaluating:   4%|▍         | 1/24 [00:00<00:08,  2.73it/s]2025-09-18 14:52:28,985 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:52:29,718 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:52:30,675 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:52:32,058 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:52:34,748 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
Evaluating:  17%|█▋        | 4/24 [00:09<00:51,  2.59s/it]2025-09-18 14:52:36,655 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:52:37,538 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:52:38,584 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:52:39,463 - INFO - HTTP Request: POST http://127.0.0.1

In [64]:
pdr_qa_result

{'context_precision': 0.5000, 'faithfulness': 0.0000, 'answer_relevancy': 0.2327, 'context_recall': 1.0000, 'answer_correctness': 0.5579, 'answer_similarity': 0.7946}

#### Ensemble Retrieval

Next let's look at ensemble retrieval!

The basic idea is as follows:

1. Obtain User Question
2. Hit the Retriever Pair
    - Retrieve Documents with BM25 Sparse Vector Retrieval
    - Retrieve Documents with Dense Vector Retrieval Method
3. Collect and "fuse" the retrieved docs based on their weighting using the Reciprocal Rank Fusion algorithm into a single ranked list.
4. Use those documents to augment our generation.

Ensure your `weights` list - the relative weighting of each retriever - sums to 1!

In [65]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever

rec_text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=75)
rec_docs = text_splitter.split_documents(base_docs)

bm25_retriever = BM25Retriever.from_documents(rec_docs, k=2)

vectorstore = FAISS.from_documents(rec_docs, embeddings)
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

ensemble_retriever = EnsembleRetriever(retrievers=[bm25_retriever, faiss_retriever], weights=[0.75, 0.25])

In [66]:
ensemble_retriever_qa_chain = create_qa_chain(ensemble_retriever)

In [67]:
ensemble_retriever_qa_chain.invoke({"question" : question})["response"].content

2025-09-18 14:55:50,438 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


"I don't know\n"

In [68]:
ensemble_qa_ragas_dataset = create_ragas_dataset(ensemble_retriever_qa_chain, eval_dataset)

100%|██████████| 4/4 [00:05<00:00,  1.37s/it]


In [69]:
ensemble_qa_ragas_dataset.to_csv("..data/ensemble_qa_ragas_dataset.csv")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 283.34ba/s]


14016

In [70]:
ensemble_qa_result = evaluate_ragas_dataset(ensemble_qa_ragas_dataset)

Evaluating:  17%|█▋        | 4/24 [00:11<01:01,  3.06s/it]2025-09-18 14:56:08,206 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:56:11,269 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
Evaluating:  21%|██        | 5/24 [00:20<01:29,  4.69s/it]2025-09-18 14:56:17,990 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:56:20,077 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:56:21,178 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:56:25,659 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:56:26,191 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:56:26,957 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2025-09-18 14:56:28,971 - INFO - HTTP Request: POST http://127.0.0.1

In [71]:
ensemble_qa_result

{'context_precision': nan, 'faithfulness': 0.6667, 'answer_relevancy': 0.4462, 'context_recall': 0.8750, 'answer_correctness': 0.5373, 'answer_similarity': 0.8346}

Observe your results in a table!

In [72]:
basic_qa_result

{'context_precision': nan, 'faithfulness': 0.3542, 'answer_relevancy': 0.2337, 'context_recall': 1.0000, 'answer_correctness': 0.5170, 'answer_similarity': 0.7954}

In [73]:
pdr_qa_result

{'context_precision': 0.5000, 'faithfulness': 0.0000, 'answer_relevancy': 0.2327, 'context_recall': 1.0000, 'answer_correctness': 0.5579, 'answer_similarity': 0.7946}

In [74]:
ensemble_qa_result

{'context_precision': nan, 'faithfulness': 0.6667, 'answer_relevancy': 0.4462, 'context_recall': 0.8750, 'answer_correctness': 0.5373, 'answer_similarity': 0.8346}

We can also zoom in on each result and find specific information about each of the questions and answers.

In [75]:
basic_qa_result

{'context_precision': nan, 'faithfulness': 0.3542, 'answer_relevancy': 0.2337, 'context_recall': 1.0000, 'answer_correctness': 0.5170, 'answer_similarity': 0.7954}

In [76]:
basic_qa_result.to_pandas()

,user_input,retrieved_contexts,response,reference,context_precision,faithfulness,answer_relevancy,context_recall,answer_correctness,answer_similarity
0,Considering the specific data structure within...,"[In contrast, LLM-as-a-judge is a general tech...","Based on the context, here’s a breakdown of th...","Given the context, an LLM-as-a-Judge would lik...",NaN,0.750000,0.934832,1.0,0.799970,0.949880
1,What specific types of complex tasks are curre...,[where LLMs are employed as evaluators for com...,I don't know.,The context indicates that LLMs are currently ...,NaN,0.000000,0.000000,1.0,0.193244,0.772975
2,Considering the specific data format and struc...,[Whether in the field of scientific research o...,I don't know.,The primary challenge in ensuring consistent m...,NaN,0.000000,0.000000,1.0,NaN,0.727402
3,Considering the specific challenges of maintai...,"[reliability, including improving consistency,...",I don't know.,The context highlights the prioritization of r...,NaN,0.666667,0.000000,1.0,0.557851,0.731405


In [77]:
pdr_qa_result

{'context_precision': 0.5000, 'faithfulness': 0.0000, 'answer_relevancy': 0.2327, 'context_recall': 1.0000, 'answer_correctness': 0.5579, 'answer_similarity': 0.7946}

In [78]:
pdr_qa_result.to_pandas()

,user_input,retrieved_contexts,response,reference,context_precision,faithfulness,answer_relevancy,context_recall,answer_correctness,answer_similarity
0,Considering the specific data structure within...,[7.1 Machine Learning\n7.1.1 NLP. LLMs have be...,"According to the context, an LLM-as-a-Judge is...","Given the context, an LLM-as-a-Judge would lik...",1.0,NaN,0.930676,1.0,NaN,0.946535
1,What specific types of complex tasks are curre...,[ABSTRACT\nAccurate and consistent evaluation ...,I don't know.,The context indicates that LLMs are currently ...,0.0,0.0,0.000000,1.0,NaN,0.772975
2,Considering the specific data format and struc...,[9.2 LLM-as-a-Judge for Data Annotation\nannot...,I don't know.,The primary challenge in ensuring consistent m...,NaN,0.0,0.000000,1.0,NaN,0.727402
3,Considering the specific challenges of maintai...,[ABSTRACT\nAccurate and consistent evaluation ...,I don't know.,The context highlights the prioritization of r...,NaN,NaN,0.000000,1.0,0.557851,0.731405


In [79]:
ensemble_qa_result

{'context_precision': nan, 'faithfulness': 0.6667, 'answer_relevancy': 0.4462, 'context_recall': 0.8750, 'answer_correctness': 0.5373, 'answer_similarity': 0.8346}

In [80]:
ensemble_qa_result.to_pandas()

,user_input,retrieved_contexts,response,reference,context_precision,faithfulness,answer_relevancy,context_recall,answer_correctness,answer_similarity
0,Considering the specific data structure within...,[language makes them well-suited for subjectiv...,"According to the provided context, an LLM-as-a...","Given the context, an LLM-as-a-Judge would lik...",NaN,1.000000,0.922845,0.833333,0.704375,0.942498
1,What specific types of complex tasks are curre...,[AI systems are evolving into highly versatile...,"According to the context, LLMs are currently b...",The context indicates that LLMs are currently ...,NaN,1.000000,0.862065,1.000000,0.349695,0.937241
2,Considering the specific data format and struc...,[Accurate and consistent evaluation is crucial...,I don't know.,The primary challenge in ensuring consistent m...,NaN,0.000000,0.000000,0.666667,NaN,0.727402
3,Considering the specific challenges of maintai...,"[learning, model selection, post-processing te...",I don't know.,The context highlights the prioritization of r...,NaN,0.666667,0.000000,1.000000,0.557851,0.731405


# Conclusion

This notebook outlines a structured approach to evaluating RAG (Retrieval-Augmented Generation) applications using RAGAS.

For best results, it is recommended to use state-of-the-art models like GPT-4 to generate high-quality ground truth (reference answers), which can then be used to assess the accuracy and relevance of responses produced by the question-answering model.